# Proyecto Final | Predicción de Readmisión Hospitalaria
## Notebook 2 — Limpieza, Encoding y Feature Engineering

**Técnicas aplicadas:** Limpieza · Encoding · StandardScaler · PCA · Train/Test Split  

---
### ¿Por qué necesitamos este paso?
- Hay columnas con **más del 40% de valores faltantes** → eliminar
- Variables categóricas como `race`, `age`, `gender` → necesitan **encoding**
- Features con rangos muy distintos → **StandardScaler**
- Alta dimensionalidad tras encoding → **PCA** reduce ruido y mejora generalización
- Split **antes** de cualquier transformación → evitar data leakage

## Tabla de Contenidos
1. [Importar librerías](#1)
2. [Cargar datos](#2)
3. [Limpieza de datos](#3)
4. [Encoding de variables categóricas](#4)
5. [Train/Test Split](#5)
6. [Estandarización con StandardScaler](#6)
7. [Reducción de dimensionalidad con PCA](#7)
8. [Visualización de componentes principales](#8)
9. [Guardar datos procesados](#9)

## 1. Importar librerías <a id='1'></a>

In [ ]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento y modelado
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

# Utilidades
import joblib
import os

sns.set_theme(style='whitegrid')
print('Librerías importadas correctamente')

## 2. Cargar datos <a id='2'></a>

Cargamos el dataset guardado en el Notebook 1.  
Ya tiene los `'?'` convertidos a NaN y el target binario creado.

In [ ]:
# Cargamos el dataset guardado en el Notebook 1
# Ya tiene los '?' convertidos a NaN y el target binario creado
df = pd.read_csv('../data/readmision_raw.csv')

print(f'Dataset cargado: {df.shape[0]:,} filas | {df.shape[1]} columnas')
df.head()

## 3. Limpieza de datos <a id='3'></a>

In [ ]:
# Eliminar columnas con >40% de nulos
# Identificadas en el Notebook 1: weight, payer_code, medical_specialty
pct_nulos = df.isnull().mean() * 100
cols_drop_nulos = pct_nulos[pct_nulos > 40].index.tolist()
print(f'Columnas eliminadas por >40% nulos: {cols_drop_nulos}')
df.drop(columns=cols_drop_nulos, inplace=True)

In [ ]:
# Eliminar columnas ID — no aportan información predictiva
df.drop(columns=['encounter_id', 'patient_nbr'], inplace=True)
print('Columnas ID eliminadas: encounter_id, patient_nbr')

In [ ]:
# Eliminar columnas de diagnóstico con demasiados valores únicos
# diag_1, diag_2, diag_3 tienen >700 categorías → no aptas para encoding directo
df.drop(columns=['diag_1', 'diag_2', 'diag_3'], inplace=True)
print('Columnas de diagnóstico eliminadas: diag_1, diag_2, diag_3')

In [ ]:
# Eliminar columna target original — ya tenemos readmitted_30
df.drop(columns=['readmitted'], inplace=True)

# Eliminar filas con nulos restantes
n_antes = len(df)
df.dropna(inplace=True)
print(f'Filas eliminadas por nulos: {n_antes - len(df):,}')
print(f'Shape final tras limpieza: {df.shape}')

## 4. Encoding de variables categóricas <a id='4'></a>

Usamos **LabelEncoder** para convertir las variables categóricas a numéricas.  
Todas las features deben ser numéricas antes de aplicar StandardScaler y PCA.

In [ ]:
# Identificar columnas categóricas
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Columnas categóricas a encodear ({len(cat_cols)}): {cat_cols}')

In [ ]:
# Aplicar LabelEncoder a todas las columnas categóricas
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print(f'Encoding completado — {len(cat_cols)} columnas transformadas')
print('Tipos de datos tras encoding:')
print(df.dtypes.value_counts())

In [ ]:
# Verificar que no quedan columnas no numéricas
print('Columnas no numéricas restantes:',
      df.select_dtypes(exclude='number').columns.tolist() or 'Ninguna')

## 5. Train/Test Split <a id='5'></a>

**Regla fundamental:** el split se hace ANTES de escalar.  
El scaler se entrena SOLO con `X_train` y luego se aplica a `X_test`  
para evitar **data leakage** (filtración de información del test al entrenamiento).

In [ ]:
# Separamos features y target
X = df.drop(columns='readmitted_30')
y = df['readmitted_30']

# Split 80/20 con stratify para mantener la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Balance train: {y_train.value_counts().to_dict()}')
print(f'Balance test:  {y_test.value_counts().to_dict()}')
print('\nstratify=y mantiene la proporción de clases en ambos conjuntos')

## 6. Estandarización con StandardScaler <a id='6'></a>

In [ ]:
scaler = StandardScaler()

# fit_transform en train — aprende media y std solo con datos de entrenamiento
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X.columns
)

# transform en test — aplica la misma transformación sin reaprender
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X.columns
)

print('Antes del escalado:')
print(X_train.describe().loc[['mean', 'std', 'min', 'max']].round(2))
print('\nDespués del escalado:')
print(X_train_scaled.describe().loc[['mean', 'std', 'min', 'max']].round(2))

## 7. Reducción de dimensionalidad con PCA <a id='7'></a>

PCA transforma las features originales en **componentes principales**,  
que son combinaciones lineales de las features originales ordenadas por varianza explicada.  
Usamos el número de componentes que retiene el **95% de la varianza**.

In [ ]:
# PCA completo para analizar cuánta varianza explica cada componente
pca_full = PCA(random_state=42).fit(X_train_scaled)

varianza_explicada = pca_full.explained_variance_ratio_
varianza_acumulada = np.cumsum(varianza_explicada)

print('Varianza explicada por componente (primeros 15):')
for i, (v, vc) in enumerate(zip(varianza_explicada[:15], varianza_acumulada[:15]), 1):
    print(f'  PC{i:02d}: {v:.3f}  ({vc:.3f} acumulado)  {"█" * int(v * 100)}')

In [ ]:
# Scree plot — varianza explicada y acumulada
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 16), varianza_explicada[:15], color='steelblue', edgecolor='white')
axes[0].set(xlabel='Componente principal', ylabel='Varianza explicada',
            title='Varianza explicada por componente', xticks=range(1, 16))

axes[1].plot(range(1, len(varianza_acumulada) + 1), varianza_acumulada,
             marker='o', color='steelblue', linewidth=2)
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95% varianza')
axes[1].axhline(y=0.90, color='orange', linestyle='--', label='90% varianza')
axes[1].set(xlabel='Número de componentes', ylabel='Varianza acumulada', title='Scree Plot')
axes[1].legend()

plt.tight_layout()
plt.show()

# Calculamos el número de componentes necesarios para 90% y 95% de varianza
n_90 = np.argmax(varianza_acumulada >= 0.90) + 1
n_95 = np.argmax(varianza_acumulada >= 0.95) + 1
print(f'Componentes para 90%: {n_90} | para 95%: {n_95}')
print(f'→ Usaremos {n_95} componentes')

In [ ]:
# Aplicar PCA con el número óptimo de componentes
# fit_transform en train, transform en test — misma regla que con el scaler
pca = PCA(n_components=n_95, random_state=42)

X_train_pca = pd.DataFrame(
    pca.fit_transform(X_train_scaled),
    columns=[f'PC{i+1}' for i in range(n_95)]
)
X_test_pca = pd.DataFrame(
    pca.transform(X_test_scaled),
    columns=[f'PC{i+1}' for i in range(n_95)]
)

print(f'Shape original:    {X_train_scaled.shape}')
print(f'Shape tras PCA:    {X_train_pca.shape}')
print(f'Varianza retenida: {pca.explained_variance_ratio_.sum():.3f}')

## 8. Visualización de componentes principales <a id='8'></a>

In [ ]:
# Scatter PC1 vs PC2 — visualizamos si las clases son separables en el espacio reducido
fig, ax = plt.subplots(figsize=(9, 6))

for clase, color, label in zip([0, 1], ['green', 'red'], ['No readmitido', 'Readmitido <30']):
    mask = y_train.values == clase
    ax.scatter(X_train_pca.loc[mask, 'PC1'], X_train_pca.loc[mask, 'PC2'],
               c=color, label=label, alpha=0.4, edgecolors='white', linewidth=0.3, s=10)

ax.set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)',
       ylabel=f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)',
       title='PCA — Separación de clases')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Guardar datos procesados <a id='9'></a>

In [ ]:
os.makedirs('../data', exist_ok=True)
os.makedirs('../scripts', exist_ok=True)

# Guardamos los datos procesados con el target para usarlos en el Notebook 3
X_train_pca.assign(target=y_train.values).to_csv('../data/train_processed.csv', index=False)
X_test_pca.assign(target=y_test.values).to_csv('../data/test_processed.csv', index=False)

# Guardamos el scaler y el pca para poder aplicarlos a datos nuevos
joblib.dump(scaler, '../scripts/scaler.pkl')
joblib.dump(pca,    '../scripts/pca.pkl')

print(f'Train: {X_train_pca.shape} | Test: {X_test_pca.shape}')
print('Scaler y PCA guardados en /scripts')

---
## Resumen del Notebook 2

| Paso | Técnica | Resultado |
|------|---------|----------|
| **Limpieza** | Eliminar cols >40% nulos + IDs + diag | Dataset reducido y limpio |
| **Encoding** | `LabelEncoder` | Todas las features numéricas |
| **Split** | `train_test_split` (80/20, stratify) | Proporciones de clase mantenidas |
| **Escalado** | `StandardScaler` | Media=0, Std=1 |
| **PCA** | `PCA(n_components=n_95)` | 95% varianza retenida |
| **Data leakage** | fit solo en train | Evitado correctamente |

---
**→ Próximo paso:** Notebook 3 — Modelado y Evaluación